# Retailrocket 电商行为分析 Agent：AI 小白半天工作坊

这份 notebook 用当前项目作为完整案例，带你从 0 理解一个数据分析 Agent 是怎么工作的。

目标不是背术语，而是看懂一条真实链路：

**用户问题 -> RAG 找资料 -> LLM 生成 SQL -> SQL 安全校验 -> SQLite 查询 -> 分析结论 -> 图表**

默认使用离线 fallback，不消耗 Gemini 或 GVMZ 额度。最后会讲如何切到真实模型。

## 0. 课堂地图

半天工作坊建议节奏：

1. 30 分钟：AI Agent 基础概念
2. 45 分钟：理解 Retailrocket 数据和 SQLite 仓库
3. 75 分钟：拆解代码，理解每个模块的职责
4. 60 分钟：亲手跑生成链路，观察 trace / RAG / SQL / 图表
5. 30 分钟：安全、评估、扩展和真实 API 接入

适合对象：完全 AI 小白，但最好能看懂一点 Python 和 SQL。

## 1. 基础概念：先用人话理解

### LLM 是什么？

LLM 可以理解成一个很会读写文字的实习生。你给它上下文和任务，它能生成回答、代码、SQL、摘要。但它不天然知道你数据库里有哪些表，也不天然保证回答一定对。

### Prompt 是什么？

Prompt 就是给模型的任务说明。好的 prompt 会告诉模型：你是谁、任务是什么、可用数据是什么、输出格式是什么、哪些事禁止做。

### RAG 是什么？

RAG = Retrieval-Augmented Generation，检索增强生成。通俗说：先翻资料，再回答。我们的资料就是 Schema 文档、指标定义、Join 示例。

### SQL Agent 是什么？

SQL Agent 是一个会把自然语言问题变成 SQL 的系统。它不是只让模型瞎写 SQL，而是会给模型 Schema，上安全校验，再执行查询。

### Multi-Agent 是什么？

Multi-Agent 不是一定要很多个大模型聊天。这里更像一条分工流水线：Router 判断问题，Retriever 找资料，Generator 写 SQL，Validator 审 SQL，Executor 查数据，Analyst 写结论，Visualization 画图。

### 幻觉是什么？

幻觉就是模型编了不存在的东西，比如数据库没有 `revenue` 字段，它却写了 `SUM(revenue)`。RAG 和 SQL 校验就是为了减少这类错误。

### 安全校验是什么？

模型生成的 SQL 不能直接执行。我们只允许 `SELECT` / `WITH`，禁止 `DROP`、`DELETE`、`UPDATE` 等危险语句，并自动加 `LIMIT`。

## 2. 先把项目加载进 notebook

这个 notebook 不复制业务代码，而是直接复用项目里的模块。这样课堂上看到的就是实际应用代码。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "agentic_analytics_demo").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("项目目录:", PROJECT_ROOT)

In [ ]:
from agentic_analytics_demo.config import AppConfig, load_environment
from agentic_analytics_demo.warehouse import ensure_demo_warehouse

load_environment()
config = AppConfig.from_env()
ensure_demo_warehouse(config.db_path, config.dataset_dir)

print("数据集目录:", config.dataset_dir)
print("SQLite 数据库:", config.db_path)
print("LLM Provider:", config.llm_provider)
print("课堂默认会强制使用 fallback，不消耗 API。")

## 3. 理解数据：Retailrocket 里有什么？

Retailrocket 是一个电商推荐系统数据集。核心行为只有三类：

- `view`：用户浏览商品
- `addtocart`：用户把商品加入购物车
- `transaction`：用户购买商品

原始 CSV 很大，所以项目启动时会构建一个 SQLite 分析仓库：

- `events`：用户行为事件
- `category_tree`：品类树
- `item_category_history`：商品品类历史
- `item_latest_category`：每个商品的最新品类
- `item_availability_history`：商品可售状态历史
- `item_latest_availability`：每个商品的最新可售状态

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(config.db_path)
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    con,
)
tables

In [ ]:
summary_sql = """
SELECT
  COUNT(*) AS event_rows,
  COUNT(DISTINCT visitorid) AS visitors,
  COUNT(DISTINCT itemid) AS items,
  MIN(event_time) AS min_time,
  MAX(event_time) AS max_time
FROM events
"""
pd.read_sql_query(summary_sql, con)

In [ ]:
pd.read_sql_query("SELECT event, COUNT(*) AS count FROM events GROUP BY event ORDER BY count DESC", con)

## 4. 代码导览：每个文件负责什么？

| 文件 | 作用 | 小白理解 |
|---|---|---|
| `warehouse.py` | 把 CSV 变成 SQLite 分析仓库 | 后厨备菜 |
| `schema_docs.py` | 写清楚表、字段、指标、Join 方法 | 给模型看的说明书 |
| `rag.py` | 从说明书里找和问题最相关的几段 | 开卷考试前翻资料 |
| `llm.py` | 调用 GVMZ/Gemini 或 fallback 生成 SQL 和分析 | 会写 SQL 的实习生 |
| `sql_guard.py` | 检查 SQL 安不安全 | 安检门 |
| `agents.py` | 把所有步骤串成 Agent 流程 | 流水线调度员 |
| `viz.py` | 把结果表变成 Plotly 图 | 图表师 |
| `app.py` | Streamlit 页面 | 前台产品界面 |

## 5. RAG：让模型先看说明书

如果直接让模型写 SQL，它可能会编不存在的字段。RAG 的作用是：根据用户问题，检索最相关的 Schema 和指标文档，然后把这些上下文塞进 prompt。

In [ ]:
from agentic_analytics_demo.rag import SchemaRAG

rag = SchemaRAG(config.chroma_dir)
rag.ensure_index()

question = "哪些品类的购买转化率最高？"
contexts = rag.retrieve(question, top_k=5)

print("RAG backend:", rag.backend)
for i, ctx in enumerate(contexts, start=1):
    print(f"\n--- Context {i}: {ctx.title} | score={ctx.score:.3f} ---")
    print(ctx.text[:350])

## 6. 如何构建 Agent：先看整条生成链路

```mermaid
flowchart TD
    A[用户自然语言问题] --> B[Semantic Router 判断请求类型]
    B --> C[Schema Retriever 检索 RAG 上下文]
    C --> D[SQL Generator 生成 SQL 和图表意图]
    D --> E[SQL Validator 安全校验]
    E --> F[Query Executor 执行 SQLite 查询]
    F --> G[Analyst Agent 生成中文洞察]
    G --> H[Visualization Agent 规范化图表配置]
    H --> I[页面展示 SQL、表格、图表、分析]
    E -- SQL 错误 --> D
```

注意：这个项目里的 Agent 不是玄学。它就是一组明确的函数节点，每个节点输入 `state`，输出更新后的 `state`。

## 7. AgentState：Agent 的记事本

`AgentState` 可以理解成 Agent 每一步传来传去的记事本。里面有：

- `question`：用户问题
- `contexts`：RAG 找到的资料
- `sql`：生成的 SQL
- `dataframe`：查询结果
- `analysis`：中文结论
- `chart_spec`：图表配置
- `trace`：每一步发生了什么
- `error`：错误信息，必要时触发修复循环

In [ ]:
from agentic_analytics_demo.agents import AgentState

demo_state: AgentState = {
    "question": "哪些商品浏览量最高？",
    "trace": []
}
demo_state

## 8. 默认离线跑完整 Agent

这里故意使用 `GeminiClient(api_key="")`，强制走 fallback。课堂上这样最稳：不用 API key，不怕额度耗尽，也能完整看到 SQL 生成链路。

In [ ]:
from agentic_analytics_demo.agents import AnalyticsAgentSystem
from agentic_analytics_demo.llm import GeminiClient

offline_llm = GeminiClient(api_key="")
system = AnalyticsAgentSystem(config.db_path, rag, offline_llm)

result = system.run("哪些商品浏览量最高？")
print("错误:", result.error or "无")
print("返回行数:", len(result.dataframe))
print("\n执行轨迹:")
for step in result.trace:
    print("-", step)

In [ ]:
print(result.sql)
result.dataframe.head(10)

In [ ]:
from agentic_analytics_demo.viz import render_plotly_chart

fig = render_plotly_chart(result.dataframe.head(20), result.chart_spec)
fig.show()

## 9. 生成链路案例库

下面把课堂要讲的例子集中跑一遍。每个问题都展示：trace、RAG 上下文标题、SQL、前几行结果、分析结论。

In [ ]:
example_questions = [
    "哪些商品浏览量最高？",
    "哪些品类的购买转化率最高？",
    "按天展示 view、addtocart、transaction 的漏斗趋势。",
    "找出高浏览但低购买的商品，适合做推荐优化。",
    "哪些品类贡献了最多交易访客？",
]

def show_chain(question: str):
    print("=" * 90)
    print("问题:", question)
    out = system.run(question)
    print("\nTrace:")
    for step in out.trace:
        print("-", step)
    print("\nRAG Contexts:", [ctx.title for ctx in out.contexts])
    print("\nSQL:\n", out.sql)
    print("\n结果预览:")
    display(out.dataframe.head(5))
    print("\n分析:", out.analysis)
    return out

outputs = [show_chain(q) for q in example_questions]

## 10. SQL 安全：为什么不能直接执行模型输出？

LLM 可能生成错误甚至危险 SQL。我们的 `sql_guard.py` 做了几件事：

- 只允许单条 SQL
- 只允许 `SELECT` / `WITH`
- 阻止 `DROP`、`DELETE`、`UPDATE`、`ALTER`、`PRAGMA` 等关键词
- 自动追加 `LIMIT`
- 用 SQLite `EXPLAIN QUERY PLAN` 做语法和表字段校验

这一步是从 demo 走向工程系统的关键。

In [ ]:
from agentic_analytics_demo.sql_guard import validate_sql

safe = validate_sql("SELECT event, COUNT(*) AS n FROM events GROUP BY event", config.db_path)
dangerous = validate_sql("DROP TABLE events", config.db_path)

print("安全 SQL 是否通过:", safe.ok, "|", safe.sql)
print("危险 SQL 是否通过:", dangerous.ok, "|", dangerous.error)

## 11. 为什么不让模型生成 Python 画图代码？

让模型生成并执行 Python 代码很危险：它可能读文件、联网、删东西，或者写出很慢的代码。

这个项目采用更安全的做法：模型只输出一个受控的 `chart_spec`，例如：

```python
{"type": "bar", "x": "itemid", "y": "views", "title": "商品浏览量排行"}
```

真正的画图由本地 `viz.py` 里的固定函数完成。这叫把“创造性”和“执行权限”分开。

## 12. 真实模型切换：GVMZ / Gemini

课堂默认不消耗 API。真实产品中可以切换到 GVMZ 或 Gemini：

- `key2.txt`：GVMZ 中转 key
- `key.txt`：Gemini 官方 key
- `.env`：也可以配置 `LLM_PROVIDER`、`GVMZ_MODEL`、`GVMZ_BASE_URL`

注意：下面只展示配置，不默认发请求。

In [ ]:
print("当前配置里的 provider:", config.llm_provider)
print("GVMZ key 是否存在:", bool(config.gvmz_api_key))
print("GVMZ 模型:", config.gvmz_model)
print("Gemini key 是否存在:", bool(config.gemini_api_key))

# 如果课堂要现场演示真实模型，可以取消下面注释：
# live_llm = GeminiClient(
#     api_key=config.gvmz_api_key,
#     model=config.gvmz_model,
#     provider="gvmz",
#     base_url=config.gvmz_base_url,
# )
# live_system = AnalyticsAgentSystem(config.db_path, rag, live_llm)
# live_system.run("哪些商品浏览量最高？")

## 13. 课堂练习

### 练习 1：改一个业务问题

把下面的问题换成你自己的问题，例如：

- 哪些商品加购量最高？
- 哪些品类浏览多但交易少？
- 哪一天 transaction 数最高？

观察 SQL 有没有符合你的预期。

In [ ]:
my_question = "哪些商品加购量最高？"
my_result = system.run(my_question)
print(my_result.sql)
my_result.dataframe.head(10)

### 练习 2：观察一次幻觉或失败

试着问一个数据里没有的东西，例如“各国家收入是多少”。

你应该观察：

- RAG 有没有给出相关表？
- SQL Validator 是否拦截了不存在字段？
- Agent 是否进入 repair loop？

这能帮助学生理解：Agent 不是永远正确，它需要边界、校验和评估。

In [ ]:
bad_question = "各国家收入是多少？"
bad_result = system.run(bad_question)
print("error:", bad_result.error)
print("trace:")
for step in bad_result.trace:
    print("-", step)
print("SQL:\n", bad_result.sql)

## 14. 其余有用知识点

### 怎么评价一个 Agent 好不好？

- 正确性：SQL 是否能执行，字段是否存在，结果是否符合业务口径
- 稳定性：同一个问题多次运行是否差异很大
- 安全性：是否会执行危险 SQL，是否会泄露 key
- 可解释性：是否能展示 trace、RAG context、SQL
- 成本：一次问题消耗多少 token，是否能用 fallback 或缓存降低成本

### 怎么迁移到另一个数据集？

最少改四处：

1. `warehouse.py`：把新数据导入 SQLite
2. `schema_docs.py`：写新表结构和业务指标
3. `llm.py`：更新 fallback SQL 和 prompt 规则
4. `app.py`：更新示例问题和页面文案

### 什么是好 prompt？

好的 prompt 会明确：角色、任务、数据上下文、禁止事项、输出格式、错误修复方式。

### 为什么要保留 fallback？

API 可能没额度、网络可能失败、模型可能临时不稳定。fallback 让 demo 和课堂始终可运行。

## 15. 总结

今天你看到的不是一个“会聊天的页面”，而是一个完整的 Agentic Analytics 系统：

- 用 SQLite 承载真实电商行为数据
- 用 RAG 给模型提供 Schema 和指标上下文
- 用 LLM/fallback 生成 SQL 和分析
- 用 SQL Guard 把风险挡在执行前
- 用 AgentState 和 LangGraph/顺序 fallback 串起节点
- 用 trace 让每一步可解释

对小白来说，最重要的理解是：**Agent 不是魔法，而是把模型放进一个有边界、有工具、有检查的工程流程里。**